# THES_SA: Sentiment-Driven Nuclear Equity Forecasting (v2.0)

**Kaggle Kernel** — Runs the full 4-phase pipeline:
- Phase 0: Data Feasibility Audit
- Phase 1: Data Collection + Preprocessing
- Phase 2: Sentiment Analysis (FinBERT)
- Phase 3: LSTM Modeling + Evaluation + SHAP

**GPU recommended** for FinBERT scoring and LSTM training.

## 1. Setup & Install Dependencies

In [ ]:
# Constrain numpy to stay compatible with Kaggle's pre-installed scipy/sklearn/tensorflow
!pip install -q "numpy<2.2.0" yfinance pandas-ta datasets shap beautifulsoup4 lxml tqdm plotly
# Verify core imports work
import numpy, scipy, sklearn, pandas, tensorflow
print(f"numpy={numpy.__version__}, scipy={scipy.__version__}, sklearn={sklearn.__version__}, tf={tensorflow.__version__}")

## 2. Clone Repository

In [ ]:
import os

# Clone the repo (use your branch)
!git clone --branch claude/nice-wozniak https://github.com/McJack3d/FIN_PP.git /kaggle/working/FIN_PP

PROJECT_ROOT = '/kaggle/working/FIN_PP/THES_SA'
os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')
!ls -la

## 3. Fix Config Paths for Kaggle

In [ ]:
import yaml

config_path = os.path.join(PROJECT_ROOT, 'config.yaml')

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update root path for Kaggle
config['paths']['root'] = PROJECT_ROOT

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Updated root path to: {PROJECT_ROOT}")

## 4. Create Required Directories

In [ ]:
from pathlib import Path

for d in ['data/raw', 'data/processed', 'data/news', 'data/hf_cache', 'results', 'cache', 'models']:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f'Created: {d}')

## 5. Phase 0 — Feasibility Audit
Downloads FNSPID dataset (~5.7 GB) and checks article coverage per ticker.

In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'data'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'sentiment'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'models'))

from data.feasibility_audit import FeasibilityAuditor

auditor = FeasibilityAuditor(config_path)
audit_report = auditor.run_audit()

if audit_report:
    print("\n=== AUDIT RESULTS ===")
    for k, v in audit_report.items():
        print(f"  {k}: {v}")

## 6. Phase 1 — Data Collection

In [ ]:
from data.quantitative_collector import QuantitativeDataCollector

quant = QuantitativeDataCollector(config_path)
quant.collect_all()

# Check output
import pandas as pd
daily = pd.read_csv('data/raw/daily_ohlcv.csv')
print(f"\nCollected {len(daily)} rows, tickers: {daily['Ticker'].unique()}")
daily.head()

In [ ]:
from data.textual_collector import TextualDataCollector

text_collector = TextualDataCollector(config_path)
text_collector.collect_all(use_newsapi=False, use_scraping=True)

# Check output
news_file = Path('data/news/all_news_combined.csv')
if news_file.exists():
    news = pd.read_csv(news_file)
    print(f"Collected {len(news)} news articles")
    news.head()
else:
    print("No news file generated — pipeline will continue without textual data")

## 7. Phase 1b — Preprocessing

In [ ]:
from data.preprocessing import FinancialDataPreprocessor, TextualDataPreprocessor

# Financial preprocessing
fin_pp = FinancialDataPreprocessor(config_path)
fin_pp.process_pipeline('daily_ohlcv.csv', 'daily_ohlcv_processed.csv')

processed = pd.read_csv('data/processed/daily_ohlcv_processed.csv')
print(f"Processed: {len(processed)} rows, {len(processed.columns)} features")
print(f"Columns: {list(processed.columns)}")
processed.head()

In [ ]:
# Textual preprocessing (if news data exists)
news_input = Path('data/news/all_news_combined.csv')
if news_input.exists():
    text_pp = TextualDataPreprocessor(config_path)
    text_pp.process_pipeline('all_news_combined.csv', 'news_processed.csv')
    
    news_proc = pd.read_csv('data/processed/news_processed.csv')
    print(f"Processed news: {len(news_proc)} articles")
    news_proc.head()
else:
    print("No news data to preprocess — skipping")

## 8. Phase 1 Summary & Visualizations

In [ ]:
from data.generate_summary import PipelineSummary

summary = PipelineSummary(config_path=config_path)
summary.generate_full_report()

In [ ]:
# Display generated plots inline
from IPython.display import Image, display
from pathlib import Path

results_dir = Path('results')
for img in sorted(results_dir.glob('*.png')):
    print(f"\n--- {img.name} ---")
    display(Image(filename=str(img), width=800))

## 9. Phase 2a — FinBERT Sentiment Scoring
**GPU accelerated** — scores each headline with ProsusAI/finbert.

In [ ]:
from sentiment.scorer import SentimentScorer

scorer = SentimentScorer(config_path)
df_scored = scorer.run_scoring_pipeline()

if not df_scored.empty:
    print(f"Scored {len(df_scored)} articles")
    print(f"Sentiment distribution:")
    print(df_scored['sentiment_score'].describe())
    df_scored.head()
else:
    print("No articles to score — check Phase 1 news collection")

## 10. Phase 2b — Sentiment Feature Engineering

In [ ]:
from sentiment.features import SentimentFeatureEngineer

engineer = SentimentFeatureEngineer(config_path)
df_merged = engineer.run_feature_pipeline()

if not df_merged.empty:
    print(f"Merged dataset: {len(df_merged)} rows, {len(df_merged.columns)} columns")
    print(f"\nColumns: {list(df_merged.columns)}")
    print(f"\nSentiment coverage: {df_merged['Sentiment_Index'].notna().mean():.1%} non-null")
    df_merged.head()
else:
    print("Feature engineering failed")

## 11. Phase 3a — LSTM Training
Trains Baseline (price-only) and Sentiment-Augmented LSTM for each ticker.

In [ ]:
from models.lstm_model import LSTMForecaster

forecaster = LSTMForecaster(config_path)
experiment_results = forecaster.run_full_experiment()

if experiment_results:
    print(f"\n=== TRAINING COMPLETE ===")
    for target, results in experiment_results.items():
        print(f"\n{target}:")
        for ticker, res in results.items():
            if 'baseline' in res and 'augmented' in res:
                print(f"  {ticker}: baseline_loss={res['baseline'].get('test_loss', 'N/A'):.6f}, "
                      f"augmented_loss={res['augmented'].get('test_loss', 'N/A'):.6f}")
else:
    print("LSTM training failed")

## 12. Phase 3b — Evaluation & Hypothesis Testing

In [ ]:
from models.evaluation import ModelEvaluator

evaluator = ModelEvaluator(config_path)
eval_report = evaluator.run_full_evaluation(experiment_results)

if eval_report:
    print("\n=== EVALUATION REPORT ===")
    import yaml as y
    print(y.dump(eval_report, default_flow_style=False))

## 13. Phase 3c — SHAP Explainability

In [ ]:
try:
    from models.explainability import SHAPAnalyzer
    
    analyzer = SHAPAnalyzer(config_path)
    analyzer.run_shap_analysis(experiment_results)
    print("SHAP analysis complete — check results/ for plots")
except Exception as e:
    print(f"SHAP analysis failed (non-critical): {e}")
    print("You can re-run this cell after debugging.")

## 14. View All Results

In [ ]:
# Display all result plots
from IPython.display import Image, display

results_dir = Path('results')
for img in sorted(results_dir.glob('*.png')):
    print(f"\n--- {img.name} ---")
    display(Image(filename=str(img), width=800))

# Show evaluation report
eval_file = results_dir / 'evaluation_report.yaml'
if eval_file.exists():
    print("\n=== EVALUATION REPORT ===")
    with open(eval_file) as f:
        print(f.read())

## 15. Download Results
Pack all outputs for download.

In [ ]:
import shutil

# Create archive of all results
shutil.make_archive('/kaggle/working/thes_sa_results', 'zip', '.', 'results')
shutil.make_archive('/kaggle/working/thes_sa_processed_data', 'zip', '.', 'data/processed')

print("Archives created:")
print("  /kaggle/working/thes_sa_results.zip")
print("  /kaggle/working/thes_sa_processed_data.zip")
print("\nDownload from the Output tab on the right.")